In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin


In [2]:
# Load dataset
data = pd.read_csv('../M1_final.csv')

# Drop unnecessary features
data.drop(columns=['TAIL_NUM', 'DEP_TIME_M', 'TAXI_OUT'], inplace=True)

# create target variable
data['is_delayed'] = np.where(
    data['DEP_DELAY'] >= 15,
    1,
    0
)
data.drop(columns=['DEP_DELAY'], inplace=True)

# Drop null rows
data.dropna(inplace=True)

def get_time_of_day(minutes):
    if 300 <= minutes < 720:        # 5:00am - 11:59am
        return 'Morning'
    elif 720 <= minutes < 1020:     # 12:00pm - 4:59pm
        return 'Afternoon'
    elif 1020 <= minutes < 1260:    # 5:00pm - 8:59pm
        return 'Evening'
    else:                           # 9:00pm - 4:59am
        return 'Night'
    

data['time_of_day'] = data['CRS_DEP_M'].apply(get_time_of_day)



# Stratified Train test split
from sklearn.model_selection import StratifiedShuffleSplit

x = data.drop('is_delayed', axis=1)
y = data['is_delayed']

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in splitter.split(data, data['is_delayed']):
    x_train = x.iloc[train_index]
    x_test = x.iloc[test_index]
    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]


class WindDirectionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='Wind'):
        self.column = column
        self.wind_dict = {
            'NNW': 340, 'CALM': 0, 'NNE': 20, 'NE': 45, 'VAR': 0, 'WSW': 230, 
            'S': 180, 'SSW': 200, 'WNW': 290, 'ESE': 115, 'N': 360, 'SW': 225, 
            'E': 90, 'W': 270, 'SSE': 155, 'ENE': 70, 'NW': 315, 'SE': 135
        }

    def fit(self, X, y=None):
        return self
    

    def transform(self, X):
        X = X.copy()
        # Map wind directions to degrees
        X['wind_deg'] = X[self.column].map(self.wind_dict)
        # Convert to radians
        X['wind_rad'] = np.deg2rad(X['wind_deg'])
        #Compute sin and cos
        X['wind_sin'] = np.sin(X['wind_rad'])
        X['wind_cos'] = np.cos(X['wind_rad'])
        # Drop original columns
        X = X.drop(columns=[self.column, 'wind_deg', 'wind_rad'])
        return X
    

class DewPointTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='Dew Point'):
        self.column = column
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()

        # Clean column
        X[self.column] = (
            X[self.column].astype(str).str.replace('\xa0', '', regex=False).str.strip()
        )
        # Convert them into numeric values
        X[self.column] = pd.to_numeric(X[self.column], errors='coerce')

        return X
    
# ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('Wind transformer', WindDirectionTransformer(column='Wind'), ['Wind']),
    ('Dew Point Transformer', DewPointTransformer(column='Dew Point'), ['Dew Point']),
    ('OrdinalEncoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ['DEST', 'OP_UNIQUE_CARRIER', 'Condition']),
    ('OneHotEncoder', OneHotEncoder(handle_unknown='ignore', drop='first'), ['time_of_day'])
], remainder='passthrough')

### Select model

In [3]:
from xgboost import XGBClassifier

xgboost_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

### create pipeline

In [4]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('Preprocessor', preprocessor),
    ('Train XGBoost', xgboost_model)
])

## Train the model

In [5]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
param_grid = {
    'Train XGBoost__n_estimators': [100, 200],          # Number of boosting rounds
    'Train XGBoost__max_depth': [3, 5, 7],               # Maximum depth of a tree
    'Train XGBoost__learning_rate': [0.05, 0.1],         # Step size shrinkage
    'Train XGBoost__subsample': [0.7, 1.0],              # Subsample ratio of the training instance
    'Train XGBoost__colsample_bytree': [0.7, 1.0]        # Subsample ratio of columns when constructing each tree
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [6]:
# 4. Instantiate and run GridSearchCV
# WARNING: This search space is very large. It will take a significant amount of time.
# Total fits = 240
print("Starting exhaustive grid search for XG Boost...")
grid_search_xgb = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring='accuracy', # We will evaluate with more metrics later
    n_jobs=-1,        # Use all available CPU cores
    verbose=2         # Show progress
)

grid_search_xgb.fit(x_train, y_train)

Starting exhaustive grid search for XG Boost...
Fitting 5 folds for each of 48 candidates, totalling 240 fits


/home/faraaz/code/minorProject/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:49:06] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/faraaz/code/minorProject/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:49:06] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/faraaz/code/minorProject/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:49:06] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/faraaz/code/minorProject/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:49:06] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END Train XGBoost__colsample_bytree=0.7, Train XGBoost__learning_rate=0.05, Train XGBoost__max_depth=3, Train XGBoost__n_estimators=100, Train XGBoost__subsample=0.7; total time=   0.3s
[CV] END Train XGBoost__colsample_bytree=0.7, Train XGBoost__learning_rate=0.05, Train XGBoost__max_depth=3, Train XGBoost__n_estimators=100, Train XGBoost__subsample=1.0; total time=   0.3s
[CV] END Train XGBoost__colsample_bytree=0.7, Train XGBoost__learning_rate=0.05, Train XGBoost__max_depth=3, Train XGBoost__n_estimators=100, Train XGBoost__subsample=0.7; total time=   0.3s
[CV] END Train XGBoost__colsample_bytree=0.7, Train XGBoost__learning_rate=0.05, Train XGBoost__max_depth=3, Train XGBoost__n_estimators=100, Train XGBoost__subsample=0.7; total time=   0.3s
[CV] END Train XGBoost__colsample_bytree=0.7, Train XGBoost__learning_rate=0.05, Train XGBoost__max_depth=3, Train XGBoost__n_estimators=100, Train XGBoost__subsample=0.7; total time=   0.5s


/home/faraaz/code/minorProject/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:49:06] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/faraaz/code/minorProject/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:49:06] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/faraaz/code/minorProject/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:49:06] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/faraaz/code/minorProject/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:49:06] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END Train XGBoost__colsample_bytree=0.7, Train XGBoost__learning_rate=0.05, Train XGBoost__max_depth=3, Train XGBoost__n_estimators=100, Train XGBoost__subsample=1.0; total time=   0.4s
[CV] END Train XGBoost__colsample_bytree=0.7, Train XGBoost__learning_rate=0.05, Train XGBoost__max_depth=3, Train XGBoost__n_estimators=100, Train XGBoost__subsample=1.0; total time=   0.4s
[CV] END Train XGBoost__colsample_bytree=0.7, Train XGBoost__learning_rate=0.05, Train XGBoost__max_depth=3, Train XGBoost__n_estimators=100, Train XGBoost__subsample=0.7; total time=   0.4s
[CV] END Train XGBoost__colsample_bytree=0.7, Train XGBoost__learning_rate=0.05, Train XGBoost__max_depth=3, Train XGBoost__n_estimators=100, Train XGBoost__subsample=1.0; total time=   0.4s
[CV] END Train XGBoost__colsample_bytree=0.7, Train XGBoost__learning_rate=0.05, Train XGBoost__max_depth=3, Train XGBoost__n_estimators=200, Train XGBoost__subsample=0.7; total time=   0.5s
[CV] END Train XGBoost__colsample_bytree=0.7,

...

[CV] END Train XGBoost__colsample_bytree=1.0, Train XGBoost__learning_rate=0.1, Train XGBoost__max_depth=7, Train XGBoost__n_estimators=100, Train XGBoost__subsample=1.0; total time=   1.0s
[CV] END Train XGBoost__colsample_bytree=1.0, Train XGBoost__learning_rate=0.1, Train XGBoost__max_depth=7, Train XGBoost__n_estimators=100, Train XGBoost__subsample=1.0; total time=   1.0s
[CV] END Train XGBoost__colsample_bytree=1.0, Train XGBoost__learning_rate=0.1, Train XGBoost__max_depth=7, Train XGBoost__n_estimators=100, Train XGBoost__subsample=1.0; total time=   1.0s
[CV] END Train XGBoost__colsample_bytree=1.0, Train XGBoost__learning_rate=0.1, Train XGBoost__max_depth=7, Train XGBoost__n_estimators=100, Train XGBoost__subsample=0.7; total time=   1.3s


/home/faraaz/code/minorProject/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:49:23] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END Train XGBoost__colsample_bytree=1.0, Train XGBoost__learning_rate=0.1, Train XGBoost__max_depth=7, Train XGBoost__n_estimators=200, Train XGBoost__subsample=0.7; total time=   1.6s
[CV] END Train XGBoost__colsample_bytree=1.0, Train XGBoost__learning_rate=0.1, Train XGBoost__max_depth=7, Train XGBoost__n_estimators=200, Train XGBoost__subsample=0.7; total time=   1.6s
[CV] END Train XGBoost__colsample_bytree=1.0, Train XGBoost__learning_rate=0.1, Train XGBoost__max_depth=7, Train XGBoost__n_estimators=200, Train XGBoost__subsample=0.7; total time=   1.6s
[CV] END Train XGBoost__colsample_bytree=1.0, Train XGBoost__learning_rate=0.1, Train XGBoost__max_depth=7, Train XGBoost__n_estimators=200, Train XGBoost__subsample=0.7; total time=   1.6s
[CV] END Train XGBoost__colsample_bytree=1.0, Train XGBoost__learning_rate=0.1, Train XGBoost__max_depth=7, Train XGBoost__n_estimators=200, Train XGBoost__subsample=1.0; total time=   1.2s
[CV] END Train XGBoost__colsample_bytree=1.0, Trai

/home/faraaz/code/minorProject/venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:49:24] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'Train XGBoost__colsample_bytree': [0.7, 1.0], 'Train XGBoost__learning_rate': [0.05, 0.1], 'Train XGBoost__max_depth': [3, 5, ...], 'Train XGBoost__n_estimators': [100, 200], ...}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('Wind transformer', ...), ('Dew Point Transformer', ...), ...]"


In [7]:
grid_search_xgb.best_score_

np.float64(0.9086925116821934)

In [8]:
xgb_model = grid_search_xgb.best_estimator_

In [9]:
y_pred = xgb_model.predict(x_test)
y_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(5764,))

In [10]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred) * 100

91.68979875086745